## Import Required Libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")


In [0]:
df_gold = df_silver.select("customer_id","city","customer_name","email","signup_date","state")


In [0]:
# ============================================================
# GOLD DIM_CUSTOMERS WRITE LOGIC
# ============================================================

# ============================================================
# FIRST RUN -> CREATE TABLE
# ============================================================



# Defensive deduplication before MERGE
df_gold = df_gold.dropDuplicates(["customer_id"])


# ============================================================
# FIRST RUN -> CREATE TABLE
# ============================================================

if not spark.catalog.tableExists(
    f"{catalog}.{gold_schema}.dim_{data_source}"
):

    (
        df_gold.write
            .format("delta")

            # Enable Change Data Feed for downstream incremental tracking
            .option("delta.enableChangeDataFeed", "true")

            # Initial table creation
            .mode("overwrite")

            .saveAsTable(
                f"{catalog}.{gold_schema}.dim_{data_source}"
            )
    )

    print(
        f"Successfully created Gold table: dim_{data_source}"
    )


# ============================================================
# INCREMENTAL RUN -> UPSERT CHANGES
# ============================================================

else:

    print(
        f"Running incremental MERGE for dim_{data_source}"
    )

    delta_table = DeltaTable.forName(
        spark,
        f"{catalog}.{gold_schema}.dim_{data_source}"
    )

    (
        delta_table.alias("target")

        .merge(
            source=df_gold.alias("source"),

            condition="""
                target.customer_id = source.customer_id
            """
        )

        # Update ONLY if business values changed
        .whenMatchedUpdate(

            condition="""
                NOT (target.customer_name <=> source.customer_name)
                OR NOT (target.city <=> source.city)
                OR NOT (target.state <=> source.state)
                OR NOT (target.email <=> source.email)
                OR NOT (target.signup_date <=> source.signup_date)
            """,

            set={

                "customer_name":
                    "coalesce(source.customer_name, target.customer_name)",

                "city":
                    "coalesce(source.city, target.city)",

                "state":
                    "coalesce(source.state, target.state)",

                "email":
                    "coalesce(source.email, target.email)",

                "signup_date":
                    "coalesce(source.signup_date, target.signup_date)"

    
            }
        )

        # Insert new customers
        .whenNotMatchedInsert(

            values={

                "customer_id":
                    "source.customer_id",

                "customer_name":
                    "source.customer_name",

                "city":
                    "source.city",

                "state":
                    "source.state",

                "email":
                    "source.email",

                "signup_date":
                    "source.signup_date"
            }
        )

        .execute()
    )

    print(
        f"Successfully merged incremental data into dim_{data_source}"
    )